# Face Lab · учебный эксперимент

Этот ноутбук показывает подготовку, обучение и отчёты на **синтетических геометрических фигурах**. Он не измеряет качество распознавания лиц. Установите `requirements-ml.txt` и Jupyter в окружение Python 3.12; выберите это окружение как kernel. Запускайте ноутбук из корня репозитория. Предобученные веса не скачиваются.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

ROOT = Path.cwd()
assert (ROOT / "scripts/create_demo_dataset.py").is_file(), "Откройте ноутбук из корня Face-Lab"
experiment = Path(tempfile.mkdtemp(prefix="face-lab-notebook-"))
print("Артефакты эксперимента:", experiment)


## 1. Создать данные

Два класса фигур нужны только для проверки кода. Каждый запуск использует отдельную временную папку, поэтому существующие данные не перезаписываются.


In [ ]:
def run(script, *args):
    subprocess.run([sys.executable, str(ROOT / "scripts" / script), *map(str, args)], check=True)

base = experiment / "patterns"
run("create_demo_dataset.py", "--out", base, "--count", 6)


## 2. Разделить выборку

Подготовка проверяет файлы и точные дубликаты, затем делит изображения внутри каждого класса с фиксированным seed. Для фотографий людей этого недостаточно: похожие кадры и одну сессию нельзя произвольно смешивать между train и test.


In [ ]:
prepared = experiment / "prepared"
run("prepare_dataset.py", "--base", base, "--out", prepared, "--seed", 42)
print((prepared / "report.json").read_text())


## 3. Обучить одну эпоху

При обучении ArcFace использует целевую метку для углового отступа. При оценке `classify()` получает только изображение. Сохраняется лучший checkpoint по validation accuracy.


In [ ]:
import torch
from ml.train import train

torch.set_num_threads(2)
out, best = train(base, prepared / "splits.csv", out_dir=experiment / "training",
                  epochs=1, batch_size=2, pretrained=False, device="cpu",
                  export_dir=experiment / "models")


## 4. Прочитать отчёт

Test вычисляется после повторной загрузки лучшей модели. Полученные числа описывают этот маленький эксперимент с фигурами; не переносите их на задачу распознавания лиц.


In [ ]:
summary = json.loads((out / "summary.json").read_text())
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Метки:", (out / "classes.json").read_text())


## 5. Продолжить исследование

- Измените seed, размер выборки и число эпох в новом запуске.
- Объясните, почему две категории могут давать высокую softmax-оценку на постороннем изображении.
- Для настоящих кропов лиц используйте `docs/training.md`; YOLO-детектор подключается отдельно.
- Веб-пример в `examples/` содержит заданную разметку и не зависит от обученной здесь модели.

Артефакты остаются в напечатанной временной папке для изучения; удалите её вручную, когда закончите.
